In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
from copy import deepcopy

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from bait.core import bait_prompts, bait_utils

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/check_contexts'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def check_context_size(contexts_all: dict):
    new_contexts_all = {}

    for file_format in bait_prompts.FILE_FORMATS:
        contexts: dict = contexts_all[file_format]

        new_contexts = {}
        for key in sorted(contexts.keys()):
            new_contexts[key] = contexts[key]

            if len(new_contexts) == bait_prompts.CONTEXT_SIZE:
                break
        
        new_contexts_all[file_format] = new_contexts
    
    return new_contexts_all

In [ ]:
def check_zero_shot(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, batch_size, out_prefix):
    zero_shot_facts, zero_shot_counters, zero_shot_others = [], [], []

    cnt = 0
    for datas_batch in container_utils.chunks(datas, batch_size):
        prompts_batch = []

        for data in datas_batch:
            question = data['question']
            prompts_batch.append(bait_prompts.get_generate_prompt(question))
        
        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        for generated_text, data in zip(generated_texts, datas_batch):
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']

            checked_data = {
                'id': data['id'],
                'question': data['question'],
                'answer_fact': data['answer_fact'],
                'answer_counter': data['answer_counter'],
                'generated_text': generated_text,
                'contexts_fact': check_context_size(data['contexts_fact']),
                'contexts_counter': check_context_size(data['contexts_counter'])
            }

            if model_utils.is_correct(generated_text, answer_fact)[1]:
                zero_shot_facts.append(checked_data)
            elif model_utils.is_correct(generated_text, answer_counter)[1]:
                zero_shot_counters.append(checked_data)
            else:
                zero_shot_others.append(checked_data)
        
        cnt += 1
        if (cnt % 50) == 0:
            print(f'check_zero_shot() {cnt} batch complet')
    print(f'check_zero_shot() {cnt} batch complet')
    
    print(f'\ncheck_zero_shot() zero_shot_fact size : {len(zero_shot_facts)}')
    print(f'check_zero_shot() zero_shot_counter size : {len(zero_shot_counters)}')
    print(f'check_zero_shot() zero_shot_other size : {len(zero_shot_others)}')

    json_utils.write_json(zero_shot_facts, f'{out_prefix}_zero_shot_fact.json')
    json_utils.write_json(zero_shot_counters, f'{out_prefix}_zero_shot_counter.json')
    json_utils.write_json(zero_shot_others, f'{out_prefix}_zero_shot_other.json')

In [ ]:
def add_prompts(question, contexts_all: dict, answer, prompts: list, answers: list):
    for file_format in bait_prompts.FILE_FORMATS:
        contexts: dict = contexts_all[file_format]

        sorted_keys = sorted(contexts.keys())
        for key in sorted_keys:
            prompts.append(bait_prompts.get_generate_prompt(question, [contexts[key]]))
            answers.append(answer)

In [ ]:
def check_and_remove_cross_inclusion(datas):
    cnt_context_fact_in_answer_counter = 0
    cnt_context_counter_in_answer_fact = 0

    cleaned_datas = []

    for data in datas:
        question = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        contexts_fact = data['contexts_fact']
        contexts_counter = data['contexts_counter']

        cleaned_contexts_fact = {}
        cleaned_contexts_counter = {}

        for file_format in bait_prompts.FILE_FORMATS:
            cleaned_contexts_fact[file_format] = {}
            cleaned_contexts_counter[file_format] = {}

            for key, context_fact in contexts_fact[file_format].items():
                if answer_counter in context_fact:
                    cnt_context_fact_in_answer_counter += 1
                else:
                    cleaned_contexts_fact[file_format][key] = context_fact
            
            for key, context_counter in contexts_counter[file_format].items():
                if answer_fact in context_counter:
                    cnt_context_counter_in_answer_fact += 1
                else:
                    cleaned_contexts_counter[file_format][key] = context_counter
        
        cleaned_data = deepcopy(data)
        cleaned_data['contexts_fact'] = cleaned_contexts_fact
        cleaned_data['contexts_counter'] = cleaned_contexts_counter
        cleaned_datas.append(cleaned_data)
    
    print(f'check_and_remove_cross_inclusion() cnt_context_fact_in_answer_counter : {cnt_context_fact_in_answer_counter}')
    print(f'check_and_remove_cross_inclusion() cnt_context_counter_in_answer_fact : {cnt_context_counter_in_answer_fact}\n')

    return cleaned_datas

In [ ]:
def check_contexts(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, batch_size=1):
    cnts_fact, cnts_counter = {}, {}
    size_per_data = len(bait_prompts.FILE_FORMATS) * bait_prompts.CONTEXT_SIZE * 2

    for i, datas_batch in enumerate(container_utils.chunks(datas, batch_size)):
        prompts_batch, answers_batch = [], []

        for data in datas_batch:
            question = data['question']
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            contexts_fact = data['contexts_fact']
            contexts_counter = data['contexts_counter']

            add_prompts(question, contexts_fact, answer_fact, prompts_batch, answers_batch)
            add_prompts(question, contexts_counter, answer_counter, prompts_batch, answers_batch)
        
        if len(prompts_batch) != (size_per_data * batch_size):
            print(f'### [ERR] check_contexts() 개수 오류')
        
        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        # for generated_text, answer in zip(generated_texts, answers_batch):
        #     is_cor = model_utils.is_correct(generated_text, answer)[1]
        #     print(f'{generated_text}\t{answer}\t{is_cor}')

        idx = 0
        for j in range(batch_size):
            # Fact
            for file_format in bait_prompts.FILE_FORMATS:
                for k in range(bait_prompts.CONTEXT_SIZE):
                    if model_utils.is_correct(generated_texts[idx], answers_batch[idx])[1]:
                        container_utils.add_str_int(cnts_fact, file_format, 1)
                    idx += 1

            # Counter
            for file_format in bait_prompts.FILE_FORMATS:
                for k in range(bait_prompts.CONTEXT_SIZE):
                    if model_utils.is_correct(generated_texts[idx], answers_batch[idx])[1]:
                        container_utils.add_str_int(cnts_counter, file_format, 1)
                    idx += 1
        
        if (i+1) % 100 == 0:
            print(f'check_contexts() {(i+1)*batch_size} checked.')
    print(f'check_contexts() {len(datas)} checked.\n')





    cnt_fact_all, cnt_counter_all = 0, 0
    for file_format in bait_prompts.FILE_FORMATS:
        cnt_fact = cnts_fact[file_format] if file_format in cnts_fact.keys() else 0
        cnt_counter = cnts_counter[file_format] if file_format in cnts_counter.keys() else 0
        percent_fact = f'{(cnt_fact / (len(datas) * bait_prompts.CONTEXT_SIZE)):.4f}'
        percent_counter = f'{(cnt_counter / (len(datas) * bait_prompts.CONTEXT_SIZE)):.4f}'
        print(f'{file_format}\t{cnt_fact}({percent_fact})\t{cnt_counter}({percent_counter})')

        cnt_fact_all += cnt_fact
        cnt_counter_all += cnt_counter
    
    size_all = len(datas) * size_per_data
    percent_fact_all = f'{(cnt_fact_all / (size_all/2)):.4f}'
    percent_counter_all = f'{(cnt_counter_all / (size_all/2)):.4f}'
    percent_all = f'{((cnt_fact_all + cnt_counter_all) / size_all):.4f}'

    print(f'All\t{cnt_fact_all}({percent_fact_all})\t{cnt_counter_all}({percent_counter_all})\t{cnt_fact_all+cnt_counter_all}({percent_all})\n')

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']

for model_name in model_names:
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)
    
    model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

    # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

    for zero_shot in ['other', 'fact', 'counter']:
        in_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
        datas = json_utils.load_json(in_file_path)

        # out_prefix = f'{out_dir}/{model_name}/check_zero_shot/bait_{model_name}_zero_shot_{zero_shot}_checked'
        # check_zero_shot(model, tokenizer, datas, 100, out_prefix)

        # cleaned_datas = check_and_remove_cross_inclusion(datas)
        # check_and_remove_cross_inclusion(cleaned_datas)
        # out_file_path = f'{out_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_checked_contexts.json'
        # json_utils.write_json(cleaned_datas, out_file_path)

        check_contexts(model, tokenizer, datas)

    del model
    del tokenizer
    common_utils.clear_gpu_memory()